# 공유오피스 3일 무료체험 결제율 하락 분석

2인이 문제 정의·분석·해석·제안 전 과정을 공동 수행했습니다.

**질문:** 이용 행동은 활발해졌는데 왜 결제율은 낮아졌을까?

2022년과 2023년 11~12월 신청 코호트를 비교합니다. 행동 지표와 경쟁사 정황으로 원인을 확정하기 어려워 미결제 사유 조사 → 실험 → CRM을 제안했습니다.

이 노트북은 원본 세 탐색 노트북의 핵심 분석을 공개용으로 정리한 것입니다. 원본 데이터와 고객별 출력은 포함하지 않습니다. [지표 정의](metric_notes.md)를 함께 확인하세요.

## 1. 입력과 데이터 품질 확인

환경변수 `OFFICE_DATA_DIR` 또는 현재 폴더의 `data/`를 사용합니다. 필요한 세 CSV와 실행 방법은 README에 있습니다. 원본 전처리 파일의 중복·식별자 누락과 결제 누락 정책을 점검합니다.

In [1]:
from pathlib import Path
import os
from analysis import (load_tables, build_cohort, summarize, fit_basic_logit,
                      save_charts, export_aggregates)

tables, input_quality = load_tables()
display(input_quality)

,table,rows,exact_duplicate_rows,missing_user_id_rows
0,register,9631,0,0
1,payment,9624,0,0
2,visit,11429,0,0


## 2. 고객 단위 코호트 구성

고객별 최신 신청일, 마지막 결제 레코드, 방문 기록을 결합합니다. 비교 월은 2022년·2023년 11~12월입니다. 전체 신청자와 방문 고객을 구분합니다.

**관찰 범위의 한계:** 방문 데이터는 신청 후 3일이나 결제 전으로 제한하지 않았습니다. 전체 방문 기록을 이용한 탐색적 비교이며, 결제 전 행동의 인과효과·미래 예측 성능으로 해석하지 않습니다.

In [2]:
cohort = build_cohort(tables)
summary = summarize(cohort)
display(summary['quality'])

{
  "cohort_size": 1144,
  "missing_payment_before_fill": 0,
  "payers_without_visit_record": 103,
  "first_visit_before_latest_application": 0,
  "first_visit_over_3_days": 0,
  "users_more_than_3_visit_days": 18
}

## 3. 전체 신청자의 월별 결제율

분모는 해당 월 전체 신청자, 분자는 그중 결제자입니다. 방문자 기준 결제율은 별도 컬럼입니다. 2023년 11월과 12월 전체 신청자 결제율은 각각 전년 동월 대비 약 26.1%p, 15.2%p 낮았습니다.

In [3]:
display(summary['monthly'].round(4))

,applicants,visitors,all_payers,visitor_payers,applicant_payment_pct,visitor_payment_pct,first_visit_pct
month,,,,,,,
2022-11,203,141,100,71,49.2611,50.3546,69.4581
2022-12,414,269,163,110,39.3720,40.8922,64.9758
2023-11,246,200,57,43,23.1707,21.5000,81.3008
2023-12,281,263,68,61,24.1993,23.1939,93.5943


![전체 신청자 결제율](assets/monthly_payment_rate.png)

## 4. 방문 단계와 방문자 결제율

방문자 결제율은 **방문 기록이 있는 결제자 / 방문 기록이 있는 신청자**입니다. 전체 결제자 수를 방문자 수로 나누지 않습니다. 첫·두 번째·세 번째 방문은 고유 방문 날짜 수에 따른 도달 여부이며, 결제일이 방문 뒤였음을 입증하는 순차 퍼널은 아닙니다.

In [4]:
display(summary['yearly'].round(4))

,applicants,visitors,all_payers,visitor_payers,second_visitors,third_visitors,applicant_payment_pct,visitor_payment_pct,first_visit_pct,second_given_first_pct,third_given_second_pct
year,,,,,,,,,,,
2022,617,410,263,181,177,69,42.6256,44.1463,66.4506,43.1707,38.9831
2023,527,463,125,104,256,119,23.7192,22.4622,87.8558,55.2916,46.4844


![방문자 결제율](assets/visitor_payment_rate.png)

## 5. 행동 지표와 같은 방문일수 구간 비교

3일 이상 방문은 표시용으로 한 구간에 묶습니다. 평균 총체류시간은 방문 고객 단위 합계의 평균입니다. 같은 구간에서도 2023년 결제율이 낮은지 확인합니다. 이는 집단 간 관련성 비교입니다.

In [5]:
display(summary['behavior'].round(4))
display(summary['strata'].round(4))

,visitors,mean_visit_days,mean_total_stay_hour,mean_used_site_count,mean_days_to_first_visit
year,,,,,
2022,410,1.6098,6.8538,1.0512,1.2098
2023,463,1.8402,8.8901,1.0778,0.3715


visitors  payers  payment_pct
year visit_days_group                               
2022 1                      233      86      36.9099
     2                      108      48      44.4444
     3                       69      47      68.1159
2023 1                      207      30      14.4928
     2                      137      35      25.5474
     3                      119      39      32.7731

## 6. 행동 변수를 포함한 기본 로지스틱 회귀

기존 기본 모형을 재현합니다. 방문일수, 총체류시간, 이용 지점수, 첫 방문 소요일, 2023년 여부를 포함합니다. 방문일수는 회귀에서 3일로 자르지 않습니다. 첫 방문 소요일 정의도 기존 회귀와 동일하게 유지합니다.

회귀는 관측한 행동 변수를 포함하고도 연도별 관련성이 남는지 확인하는 탐색적 분석입니다. 개별 행동 변수의 부호를 개선 액션의 인과효과로 읽지 않습니다.

In [6]:
logit, logit_info = fit_basic_logit(cohort)
display(logit_info)
display(logit.round(6))

{
  "nobs": 873,
  "converged": true,
  "excluded_missing_or_infinite": 0
}

,coefficient,odds_ratio,p_value,ci_low,ci_high
const,-1.844718,0.158070,0.000000,0.079072,0.315990
visit_days,0.841154,2.319042,0.000000,1.793256,2.998990
total_stay_hour,-0.042399,0.958488,0.005547,0.930195,0.987641
used_site_count,0.220712,1.246964,0.384870,0.757971,2.051425
days_to_first_visit,0.252460,1.287189,0.018258,1.043755,1.587398
year_2023,-0.983999,0.373813,0.000000,0.264037,0.529231


연도 오즈비는 약 **0.374**, 95% 신뢰구간은 약 **0.264~0.529**입니다. 확률이 0.374배라는 뜻이나 연도가 결제 하락의 원인이라는 뜻은 아닙니다. 실제 결제시각과 관찰기간 제한을 보완해야 합니다.

## 7. 경쟁사 검토와 후속 검증

가격·지점 접근성·프로모션을 검토했지만 자료의 시점과 고객별 실제 노출이 부족했습니다. 경쟁사를 원인으로 확정하지 않았습니다.

| 단계 | 제안 |
|---|---|
| 미결제 사유 조사 | 가격·위치·서비스·이용 필요·경쟁사·상품 정보 등을 복수 선택 후 핵심 사유 확인 |
| 로그 연결 | 응답, 안내 노출, 실제 결제일, 같은 기간 행동 연결 |
| 안내 실험 | 일반 안내와 행동 기반 개인화 안내의 무작위 비교 |
| 혜택 실험 | 20% 할인과 비가격 혜택의 상대 효과·비용 비교 |
| CRM | 검증된 대응안을 해당 고객군에 적용하고 효과 지속 점검 |

투표·실험·CRM은 제안 단계입니다. 실험은 제안한 대응안의 효과를 검증하며, 과거 하락 원인 전체를 확정하지 않습니다. 혜택을 주지 않은 경우 대비 효과는 별도 대조군이 필요합니다.

![후속 검증 제안](assets/preview.png)

## 8. 공개 가능한 결과 저장

아래 코드는 집계값과 그래프만 저장합니다. 고객별 레코드와 식별자를 내보내지 않습니다.

In [7]:
save_charts(summary)
_ = export_aggregates(summary, logit, logit_info)
print('집계 결과와 그래프 저장 완료')

집계 결과와 그래프 저장 완료
